In [6]:
import numpy as np
from google.cloud import storage
import io
from orchestrator import Orchestrator
import cProfile
import pstats
import io
import sys
sys.path.append("..")

from creds import MAPPER_URL, REDUCER_URL

In [7]:
bucket_name = "jack-fall2024"
block_size = 10

In [8]:
# Generate Matrices
Matrix_A = np.random.randint(0, 10, (40, 40))  
Matrix_B = np.random.randint(0, 10, (40, 40)) 

# Initialize Google Cloud Storage client
client = storage.Client()
bucket = client.bucket(bucket_name)

# Function to upload matrix to GCS directly from bytes
def upload_matrix_to_gcs(matrix, matrix_name):
    # Convert the matrix to bytes
    matrix_bytes = io.BytesIO()
    np.save(matrix_bytes, matrix)
    matrix_bytes.seek(0)  # Move the pointer to the start of the byte stream

    # Define the GCS blob (object) name
    blob = bucket.blob(f"{matrix_name}.npy")
    
    # Upload the matrix bytes to GCS
    blob.upload_from_file(matrix_bytes, content_type='application/octet-stream')
    print(f"Uploaded {matrix_name} to {bucket_name}.")

# Upload both matrices
upload_matrix_to_gcs(Matrix_A, "Matrix_A")
upload_matrix_to_gcs(Matrix_B, "Matrix_B")




Uploaded Matrix_A to jack-fall2024.
Uploaded Matrix_B to jack-fall2024.


In [9]:

print(np.matmul(Matrix_A, Matrix_B))

[[859 777 703 ... 776 753 642]
 [909 829 766 ... 889 751 746]
 [894 934 765 ... 933 784 829]
 ...
 [927 844 811 ... 888 710 842]
 [789 792 721 ... 775 773 793]
 [812 858 828 ... 836 786 828]]


In [10]:

shape = (40,40)
Matrix_A = "Matrix_A.npy"
Matrix_B = "Matrix_B.npy"

# orch = Orchestrator(Matrix_A, Matrix_B, shape, shape, block_size, MAPPER_URL, REDUCER_URL, bucket_name)
# map = orch.orchestrate_matrix_multiplication()
# print(map)


orch = Orchestrator(Matrix_A, Matrix_B, shape, shape, block_size, MAPPER_URL, REDUCER_URL, bucket_name)

# Profile the orchestrate_matrix_multiplication function
pr = cProfile.Profile()
pr.enable()
orch.orchestrate_matrix_multiplication()
pr.disable()

# Print the profiling results
s = io.StringIO()
ps = pstats.Stats(pr, stream=s)
ps.sort_stats('cumulative')
ps.print_stats()

print(s.getvalue())

         1189473 function calls (1187544 primitive calls) in 82.310 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000   82.316   41.158 /home/jackyeung99/classes/Engineering_Cloud_Computing/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3541(run_code)
        2    0.000    0.000   82.316   41.158 {built-in method builtins.exec}
        1    0.021    0.021   82.311   82.311 /home/jackyeung99/classes/Engineering_Cloud_Computing/Assignment03/orchestrator.py:123(orchestrate_matrix_multiplication)
     6663    0.013    0.000   69.071    0.010 /usr/lib/python3.10/threading.py:589(wait)
    34080   69.052    0.002   69.052    0.002 {method 'acquire' of '_thread.lock' objects}
     6639    0.023    0.000   69.049    0.010 /usr/lib/python3.10/threading.py:288(wait)
     6562    0.025    0.000   69.043    0.011 /usr/lib/python3.10/concurrent/futures/_base.py:201(as_completed)
        1